# 03. FMA REAL 오디오 검증

최종 매핑된 FMA REAL 296곡의 다운로드 상태와 파일 무결성을 확인한다. 매핑 행 수와 파일 수, `track_id`, 파일 크기, 디코딩 여부, 재생시간을 차례로 점검한 뒤 곡별 결과를 CSV로 저장한다.


In [21]:
from pathlib import Path
import subprocess
import shutil
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

MAPPING_PATH = PROJECT_ROOT / "data/metadata/fma_real_mapping.csv"
AUDIO_DIR = PROJECT_ROOT / "data/raw/FMA/selected_30s"
REPORT_PATH = PROJECT_ROOT / "data/metadata/fma_real_audio_validation.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MAPPING_PATH:", MAPPING_PATH)
print("AUDIO_DIR   :", AUDIO_DIR)


PROJECT_ROOT: <PROJECT_ROOT>
MAPPING_PATH: <PROJECT_ROOT>/data/metadata/fma_real_mapping.csv
AUDIO_DIR   : <PROJECT_ROOT>/data/raw/FMA/selected_30s


**결과:** 프로젝트 루트와 매핑 CSV, REAL 오디오 폴더 경로가 설정되었다.

## 1. 매핑 행과 MP3 파일 수

`fma_real_mapping.csv`의 행 수, 고유 `track_id` 수와 실제 MP3 파일 수를 비교한다.


In [22]:
mapping = pd.read_csv(MAPPING_PATH)
audio_files = sorted(AUDIO_DIR.rglob("*.mp3"))

print("===== FMA REAL AUDIO CHECK =====")
print("Mapping rows      :", len(mapping))
print("Unique track_id   :", mapping["track_id"].nunique())
print("Downloaded MP3    :", len(audio_files))


===== FMA REAL AUDIO CHECK =====
Mapping rows      : 296
Unique track_id   : 296
Downloaded MP3    : 296


**결과:** 매핑은 296행이며 `track_id`도 296개로 모두 고유하다. 오디오 폴더에서 찾은 MP3 역시 296개다.

## 2. Track ID 일치 여부

매핑에서 기대한 ID 집합과 MP3 파일명에서 읽은 ID 집합을 비교한다.


In [23]:
expected_ids = set(mapping["track_id"].astype(int).tolist())
downloaded_ids = {int(path.stem) for path in audio_files}

missing_ids = sorted(expected_ids - downloaded_ids)
extra_ids = sorted(downloaded_ids - expected_ids)

print("===== TRACK ID MATCH CHECK =====")
print("Expected :", len(expected_ids))
print("Found    :", len(downloaded_ids))
print("Missing  :", len(missing_ids))
print("Extra    :", len(extra_ids))

if missing_ids:
    print("\nMissing track IDs:")
    print(missing_ids)

if extra_ids:
    print("\nExtra track IDs:")
    print(extra_ids)


===== TRACK ID MATCH CHECK =====
Expected : 296
Found    : 296
Missing  : 0
Extra    : 0


**결과:** 기대한 ID와 실제 파일 ID가 각각 296개로 같고, 누락과 추가 파일은 모두 0개다.

## 3. 파일 크기 검사

0 byte 파일과 100 KB 미만 파일을 집계하고 전체 크기 분포를 확인한다.


In [24]:
file_check = []

for path in audio_files:
    file_check.append({
        "track_id": int(path.stem),
        "path": str(path.relative_to(PROJECT_ROOT)),
        "size_bytes": path.stat().st_size,
    })

file_check = pd.DataFrame(file_check)

zero_byte_count = int((file_check["size_bytes"] == 0).sum())
small_files = file_check[file_check["size_bytes"] < 100_000].copy()

print("0 byte files   :", zero_byte_count)
print("100KB 미만 파일:", len(small_files))

display(file_check["size_bytes"].describe())

if len(small_files):
    display(small_files)


0 byte files   : 0
100KB 미만 파일: 0


count    2.960000e+02
mean     1.021890e+06
std      2.238527e+05
min      2.405400e+05
25%      9.608410e+05
50%      1.159154e+06
75%      1.201366e+06
max      1.203312e+06
Name: size_bytes, dtype: float64

**결과:** 0 byte와 100 KB 미만 파일은 모두 0개다. 파일 크기는 최소 240,540 byte, 평균 약 1.02 MB, 최대 1,203,312 byte였다.

## 4. MP3 디코딩과 재생시간

`ffprobe`를 우선 사용하고, 사용할 수 없으면 `librosa`로 각 파일의 디코딩 여부와 재생시간을 확인한다.


In [6]:
FFPROBE_AVAILABLE = shutil.which("ffprobe") is not None

print("ffprobe available:", FFPROBE_AVAILABLE)

if not FFPROBE_AVAILABLE:
    try:
        import librosa
        print("librosa available: True")
    except ImportError:
        print("librosa available: False")
        raise RuntimeError(
            "ffprobe 또는 librosa 중 하나가 필요합니다. "
            "Mac에서는 `brew install ffmpeg` 또는 현재 conda 환경에 librosa를 설치하세요."
        )


ffprobe available: True


**결과:** 실행 환경에서 `ffprobe`를 사용할 수 있어 이후 296개 파일의 디코딩과 재생시간 측정에 이를 사용한다.


In [25]:
def probe_audio_duration(path: Path):
    """Return (ok, duration_sec, error_message)."""
    if FFPROBE_AVAILABLE:
        cmd = [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path),
        ]
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            return False, np.nan, result.stderr.strip()

        try:
            duration = float(result.stdout.strip())
            return True, duration, ""
        except ValueError:
            return False, np.nan, "duration 값을 float로 변환하지 못함"

    else:
        try:
            duration = float(librosa.get_duration(path=str(path)))
            return True, duration, ""
        except Exception as e:
            return False, np.nan, f"{type(e).__name__}: {e}"


audio_validation = []

for i, path in enumerate(audio_files, start=1):
    ok, duration_sec, error = probe_audio_duration(path)

    audio_validation.append({
        "track_id": int(path.stem),
        "path": str(path.relative_to(PROJECT_ROOT)),
        "size_bytes": path.stat().st_size,
        "decode_ok": ok,
        "duration_sec": duration_sec,
        "decode_error": error,
    })

    if i % 50 == 0 or i == len(audio_files):
        print(f"checked: {i}/{len(audio_files)}")

audio_validation = pd.DataFrame(audio_validation)

print("\n===== DECODE CHECK =====")
print("Decode success:", int(audio_validation["decode_ok"].sum()))
print("Decode failed :", int((~audio_validation["decode_ok"]).sum()))


checked: 50/296
checked: 100/296
checked: 150/296
checked: 200/296
checked: 250/296
checked: 296/296

===== DECODE CHECK =====
Decode success: 296
Decode failed : 0


**결과:** 296개 MP3를 모두 검사했으며 디코딩 성공 296개, 실패 0개였다.


In [26]:
decode_failures = audio_validation[~audio_validation["decode_ok"]].copy()

if len(decode_failures):
    print("===== DECODE FAILURES =====")
    display(decode_failures)
else:
    print("모든 MP3 파일이 정상적으로 디코딩되었습니다.")

print("\n===== DURATION SUMMARY =====")
display(audio_validation["duration_sec"].describe())


모든 MP3 파일이 정상적으로 디코딩되었습니다.

===== DURATION SUMMARY =====


count    296.000000
mean      29.999641
std        0.012909
min       29.988571
25%       29.988571
50%       29.988571
75%       30.014694
max       30.014694
Name: duration_sec, dtype: float64

**결과:** 디코딩 실패는 없었다. 296곡의 재생시간은 평균 29.999641초이며 최소 29.988571초, 최대 30.014694초다.

### 4.1 교체 이력 확인

현재 출력은 교체 작업까지 반영된 최종 상태다. 아래에는 초기 점검에서 크기가 약 1.6 KB였던 두 후보(148786, 148788)를 정상 후보로 바꾼 과정을 남겼다.


In [12]:
%pip install remotezip

  Using cached remotezip-0.12.6-py3-none-any.whl.metadata (7.3 kB)
Using cached remotezip-0.12.6-py3-none-any.whl (8.6 kB)
Note: you may need to restart the kernel to use updated packages.


**결과:** `remotezip` 0.12.6이 설치되었고, 나머지 의존성은 기존 환경에 이미 있었다.


In [13]:
from remotezip import RemoteZip

URL = "https://os.unil.cloud.switch.ch/fma/fma_large.zip"

check_ids = [
    148786,  # Loved Ones - 현재 후보
    148802,  # Loved Ones - 대체 후보
    148788,  # Strongbreeze - 현재 후보
    148804,  # Strongbreeze - 대체 후보
]

with RemoteZip(URL) as z:
    for track_id in check_ids:
        folder = f"{track_id // 1000:03d}"
        filename = f"{track_id:06d}.mp3"
        member = f"fma_large/{folder}/{filename}"

        print("=" * 60)
        print("track_id:", track_id)

        try:
            info = z.getinfo(member)

            print("file size      :", info.file_size)
            print("compressed size:", info.compress_size)

        except KeyError:
            print("ZIP에 파일 없음")

track_id: 148786
file size      : 1604
compressed size: 433
track_id: 148802
file size      : 1201147
compressed size: 1192888
track_id: 148788
file size      : 1608
compressed size: 435
track_id: 148804
file size      : 1201151
compressed size: 1196334


**결과:** 원래 선택된 148786과 148788은 각각 1,604 byte와 1,608 byte였고, 대체 후보 148802와 148804는 각각 1,201,147 byte와 1,201,151 byte였다.

### 4.2 대체 파일 다운로드

원격 FMA archive에서 대체 후보 148802와 148804를 별도 점검 폴더로 내려받는다.


In [14]:
from remotezip import RemoteZip
from pathlib import Path
import shutil

URL = "https://os.unil.cloud.switch.ch/fma/fma_large.zip"

replacement_ids = [148802, 148804]

output_dir = PROJECT_ROOT / "data/raw/FMA/replacement_check"

with RemoteZip(URL) as z:
    for track_id in replacement_ids:

        folder = f"{track_id // 1000:03d}"
        filename = f"{track_id:06d}.mp3"

        member = f"fma_large/{folder}/{filename}"
        dst = output_dir / folder / filename

        dst.parent.mkdir(parents=True, exist_ok=True)

        with z.open(member) as src, open(dst, "wb") as out:
            shutil.copyfileobj(src, out)

        print(
            track_id,
            "->",
            dst,
            "|",
            dst.stat().st_size,
            "bytes"
        )

148802 -> <PROJECT_ROOT>/data/raw/FMA/replacement_check/148/148802.mp3 | 1201147 bytes
148804 -> <PROJECT_ROOT>/data/raw/FMA/replacement_check/148/148804.mp3 | 1201151 bytes


**결과:** 두 대체 파일을 `replacement_check/148/` 아래에 저장했다. 저장 크기는 각각 1,201,147 byte와 1,201,151 byte다.

### 4.3 대체 파일 검증

다운로드한 두 파일의 디코딩 여부와 재생시간을 확인한다.


In [15]:
replacement_results = []

for track_id in replacement_ids:

    folder = f"{track_id // 1000:03d}"
    path = (
        PROJECT_ROOT
        / "data/raw/FMA/replacement_check"
        / folder
        / f"{track_id:06d}.mp3"
    )

    ok, duration_sec, error = probe_audio_duration(path)

    replacement_results.append({
        "track_id": track_id,
        "size_bytes": path.stat().st_size,
        "decode_ok": ok,
        "duration_sec": duration_sec,
        "error": error
    })

replacement_results = pd.DataFrame(replacement_results)

display(replacement_results)

,track_id,size_bytes,decode_ok,duration_sec,error
0,148802,1201147,True,29.988571,
1,148804,1201151,True,29.988571,


**결과:** 148802와 148804는 모두 디코딩되었고 재생시간은 각각 29.988571초였다.

### 4.4 매핑 수정

기존 ID 148786, 148788을 대체 ID 148802, 148804로 바꾸고 매핑 CSV를 다시 저장한다.


In [17]:
mapping_path = PROJECT_ROOT / "data/metadata/fma_real_mapping.csv"

mapping = pd.read_csv(mapping_path)

replacement_map = {
    148786: 148802,
    148788: 148804,
}

for old_id, new_id in replacement_map.items():
    mask = mapping["track_id"] == old_id

    print(
        old_id,
        "->",
        new_id,
        "| rows:",
        mask.sum()
    )

    mapping.loc[mask, "track_id"] = new_id

mapping.to_csv(
    mapping_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", mapping_path)
print("Rows:", len(mapping))
print("Unique track_id:", mapping["track_id"].nunique())

148786 -> 148802 | rows: 1
148788 -> 148804 | rows: 1
Saved: <PROJECT_ROOT>/data/metadata/fma_real_mapping.csv
Rows: 296
Unique track_id: 296


**결과:** 두 기존 ID가 각각 한 행씩 대체되었다. 저장 후 매핑은 296행이며 `track_id` 296개가 모두 고유하다.

### 4.5 검증 파일 반영

점검을 마친 두 대체 파일을 최종 `selected_30s` 폴더로 복사한다.


In [18]:
import shutil

replacements = {
    148802: 148802,
    148804: 148804,
}

for track_id in replacements:

    folder = f"{track_id // 1000:03d}"
    filename = f"{track_id:06d}.mp3"

    src = (
        PROJECT_ROOT
        / "data/raw/FMA/replacement_check"
        / folder
        / filename
    )

    dst = (
        PROJECT_ROOT
        / "data/raw/FMA/selected_30s"
        / folder
        / filename
    )

    dst.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src, dst)

    print("Copied:", src, "->", dst)

Copied: <PROJECT_ROOT>/data/raw/FMA/replacement_check/148/148802.mp3 -> <PROJECT_ROOT>/data/raw/FMA/selected_30s/148/148802.mp3
Copied: <PROJECT_ROOT>/data/raw/FMA/replacement_check/148/148804.mp3 -> <PROJECT_ROOT>/data/raw/FMA/selected_30s/148/148804.mp3


**결과:** 대체 파일 148802와 148804를 `selected_30s/148/` 폴더에 복사했다.


In [20]:
#이전에 깨진 파일 제거

bad_ids = [148786, 148788]

for track_id in bad_ids:

    folder = f"{track_id // 1000:03d}"
    filename = f"{track_id:06d}.mp3"

    path = (
        PROJECT_ROOT
        / "data/raw/FMA/selected_30s"
        / folder
        / filename
    )

    if path.exists():
        path.unlink()
        print("Removed:", path)

**결과:** 기존 ID 148786과 148788이 이미 폴더에 없어 삭제 메시지는 출력되지 않았다.

교체 뒤 1~4절을 다시 실행했으며, 현재 노트북 상단의 결과는 파일 296개가 모두 통과한 최종 상태다.


## 5. 재생시간 이상값

0초 이하 또는 31초 초과 파일과 10초 미만 파일을 별도로 집계한다.


In [27]:
invalid_duration = audio_validation[
    (audio_validation["decode_ok"]) &
    (
        (audio_validation["duration_sec"] <= 0) |
        (audio_validation["duration_sec"] > 31)
    )
].copy()

very_short = audio_validation[
    (audio_validation["decode_ok"]) &
    (audio_validation["duration_sec"] < 10)
].copy()

print("0초 이하 또는 31초 초과:", len(invalid_duration))
print("10초 미만             :", len(very_short))

if len(invalid_duration):
    display(invalid_duration)

if len(very_short):
    print("\n===== 10초 미만 파일 =====")
    display(very_short)


0초 이하 또는 31초 초과: 0
10초 미만             : 0


**결과:** 0초 이하 또는 31초 초과 파일은 0개이며, 10초 미만 파일도 0개다.

## 6. 매핑과 오디오 검사 결과 결합

매핑 정보와 파일 경로, 크기, 디코딩 여부, 재생시간을 `track_id` 기준으로 일대일 결합한다.


In [28]:
real_validation = mapping.merge(
    audio_validation,
    on="track_id",
    how="left",
    validate="one_to_one",
)

real_validation["file_exists"] = real_validation["path"].notna()
real_validation["size_ok"] = real_validation["size_bytes"].fillna(0) >= 100_000

display(real_validation.head())


,original_audio,genre,track_id,title,artist,genre_top,license,duration,subset,candidate_count,license_allowed,license_fallback,genre_match,path,size_bytes,decode_ok,duration_sec,decode_error,file_exists,size_ok
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,140932,"10,000 People Chanting, ""I'm an Individual""",Nihilore,Electronic,Creative Commons Attribution,372,medium,1,True,False,True,data/raw/FMA/selected_30s/140/140932.mp3,1200775,True,29.988571,,True,True
1,1984 - Punk Rock Opera,Rock,149410,1984,Punk Rock Opera,Rock,Attribution,200,medium,2,True,False,True,data/raw/FMA/selected_30s/149/149410.mp3,1203312,True,30.014694,,True,True
2,2 (Wasn't There) - Isle of Pine,Rock,66449,2 (Wasn't There),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,108,medium,1,True,False,True,data/raw/FMA/selected_30s/066/066449.mp3,984398,True,30.014694,,True,True
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,114244,2Much (Andy Spinelli & Alex Sánchez House Edit),Tentacles,Electronic,Attribution,486,medium,1,True,False,True,data/raw/FMA/selected_30s/114/114244.mp3,1201308,True,29.988571,,True,True
4,3 am West End - statusq,Electronic,112378,3 am West End,statusq,Electronic,Attribution,291,medium,1,True,False,True,data/raw/FMA/selected_30s/112/112378.mp3,600853,True,29.988571,,True,True


**결과:** 매핑과 오디오 검사 결과가 일대일로 결합되었다. 출력된 앞의 5개 행은 모두 파일이 존재하고 100 KB 이상이며 디코딩이 가능하다.

## 7. 최종 QC

지금까지의 검사 수치를 한 표로 모으고 필수 조건의 통과 여부를 계산한다.


In [29]:
qc_summary = pd.DataFrame({
    "check": [
        "mapping_rows",
        "unique_track_id",
        "downloaded_mp3",
        "missing_track_id",
        "extra_track_id",
        "zero_byte_files",
        "under_100kb_files",
        "decode_success",
        "decode_failed",
        "duration_outside_0_to_31",
        "under_10sec_files",
    ],
    "value": [
        len(mapping),
        mapping["track_id"].nunique(),
        len(audio_files),
        len(missing_ids),
        len(extra_ids),
        zero_byte_count,
        len(small_files),
        int(audio_validation["decode_ok"].sum()),
        int((~audio_validation["decode_ok"]).sum()),
        len(invalid_duration),
        len(very_short),
    ]
})

display(qc_summary)

all_core_checks_pass = (
    len(mapping) == 296
    and mapping["track_id"].nunique() == 296
    and len(audio_files) == 296
    and len(missing_ids) == 0
    and len(extra_ids) == 0
    and zero_byte_count == 0
    and int((~audio_validation["decode_ok"]).sum()) == 0
)

print("===== FINAL RESULT =====")
print("Core QC PASS:", all_core_checks_pass)


,check,value
0,mapping_rows,296
1,unique_track_id,296
2,downloaded_mp3,296
3,missing_track_id,0
4,extra_track_id,0
5,zero_byte_files,0
6,under_100kb_files,0
7,decode_success,296
8,decode_failed,0
9,duration_outside_0_to_31,0


===== FINAL RESULT =====
Core QC PASS: True


**결과:** 매핑·파일·고유 ID는 각각 296개다. 누락, 추가 파일, 0 byte, 100 KB 미만, 디코딩 실패, 재생시간 이상값은 모두 0개이며 `Core QC PASS`는 `True`다.

## 8. 검증 결과 저장

곡별 매핑 정보와 오디오 검사 결과를 CSV로 저장한다.


In [30]:
real_validation.to_csv(
    REPORT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", REPORT_PATH)
print("Rows :", len(real_validation))


Saved: <PROJECT_ROOT>/data/metadata/fma_real_audio_validation.csv
Rows : 296


**결과:** `data/metadata/fma_real_audio_validation.csv`에 검증 결과 296행을 저장했다.

## 다음 단계

검증을 통과한 FMA REAL 296곡과 Clean TTA FAKE 3,162개를 합쳐 `master_manifest.csv`를 만들고, `original_audio` 단위로 학습·검증·테스트 데이터를 나눈다.


## 정리

- 매핑 및 MP3 파일: 각 296개
- 누락·추가 파일: 0개
- 0 byte·100 KB 미만 파일: 0개
- 디코딩 성공: 296개
- 재생시간 이상값: 0개
- 최종 판정: `Core QC PASS = True`
